In [ ]:
"""
sandbox_bweight_time.ipynb

A sandbox to summarize bweights over time.

Author: Stellina X. Ao
Created: 2026-07-25
Last Modified: 2026-07-25
Python Version: 3.11.14
"""


import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
subj_id = "MR82"
sess_id = "20251027_152036"

## init

In [ ]:
from sg.models import make_tre, Encoder, StrategyEncoder

encoder = make_tre(Encoder)(
    subj_id,
    sess_id,
    norm=True,
    stepsize_s=0.1,
)

encoder_mb = make_tre(StrategyEncoder)(
    subj_id,
    sess_id,
    norm=True,
    stepsize_s=0.1,
    strategy_filter="mb",
)

encoder_mf = make_tre(StrategyEncoder)(
    subj_id,
    sess_id,
    norm=True,
    stepsize_s=0.1,
    strategy_filter="mf",
)

encoder.fit_encoder()
encoder_mb.fit_encoder()
encoder_mf.fit_encoder()

encoder.verify()
encoder_mb.verify()
encoder_mf.verify()

In [ ]:
encoder.fit_encoder()
encoder.verify()

## bweight traces

In [ ]:
import numpy as np


def get_bw_stats(encoder):
    bw_mean = {}
    bw_std = {}
    for reg in encoder.regions:
        bw_mean[reg] = np.abs(
            encoder.encoder_weights[:, encoder.reg_idxs[reg], :]
        ).mean(axis=1)
        bw_std[reg] = np.abs(encoder.encoder_weights[:, encoder.reg_idxs[reg], :]).std(
            axis=1
        )
    return bw_mean, bw_std

In [ ]:
from core.data import tv_vals
from utils.colors import colors_strategy


def plot_bw_traces(encoders, bw_stats):
    encoder = encoders["full"]
    fig, axes = plt.subplots(
        nrows=len(encoder.tv_keys) + 2,
        ncols=len(encoder.regions),
        figsize=(6, 12),
        sharex=True,
        sharey=True,
        tight_layout=True,
    )

    for j, reg in enumerate(encoder.regions):
        i = 0
        for regr in encoder.tv_keys:
            if regr != "response_prev":
                for k, (bw_mean, bw_std) in bw_stats.items():
                    ax = axes[i][j]
                    try:
                        m = bw_mean[reg][
                            :, encoders[k].dm_idxs[f"{regr}_{tv_vals[regr][0]}"]
                        ]
                        s = bw_std[reg][
                            :, encoders[k].dm_idxs[f"{regr}_{tv_vals[regr][0]}"]
                        ]
                    except KeyError:
                        try:
                            m = bw_mean[reg][
                                :, encoders[k].dm_idxs[f"{regr}_{tv_vals[regr][1]}"]
                            ]
                            s = bw_std[reg][
                                :, encoders[k].dm_idxs[f"{regr}_{tv_vals[regr][1]}"]
                            ]
                        except KeyError:
                            continue

                    ax.plot(
                        encoders[k].tbin_centers,
                        m,
                        color=colors_strategy[k],
                        label=f"{regr}, {k}",
                    )
                    ax.fill_between(
                        encoders[k].tbin_centers,
                        m - s,
                        m + s,
                        color=colors_strategy[k],
                        alpha=0.25,
                    )
                    ax.axhline(y=0, linewidth=0.5, color="k")
                    ax.axvline(x=0, linewidth=0.5, color="k")

                    ax.set_ylim([-0.2, 0.6])
                    ax.legend(loc="upper right")
                    if i == len(encoder.tv_keys) + 2 - 1:
                        ax.set_xlabel("Trial Time (s)")
                    if j == 0:
                        ax.set_ylabel(r"$\beta$")
                    if i == 0:
                        ax.set_title(reg)
                i += 1
            else:
                for val in tv_vals[regr]:
                    for k, (bw_mean, bw_std) in bw_stats.items():
                        ax = axes[i][j]
                        try:
                            m = bw_mean[reg][:, encoders[k].dm_idxs[f"{regr}_{val}"]]
                        except KeyError:
                            continue
                        s = bw_std[reg][:, encoders[k].dm_idxs[f"{regr}_{val}"]]

                        ax.plot(
                            encoder.tbin_centers,
                            m,
                            color=colors_strategy[k],
                            label=f"{regr}_{val}, {k}",
                        )
                        ax.fill_between(
                            encoder.tbin_centers,
                            m - s,
                            m + s,
                            color=colors_strategy[k],
                            alpha=0.25,
                        )
                        ax.axhline(y=0, linewidth=0.5, color="k")
                        ax.axvline(x=0, linewidth=0.5, color="k")

                        ax.set_ylim([-0.2, 0.6])
                        ax.legend(loc="upper right")
                        if i == len(encoder.tv_keys) + 2 - 1:
                            ax.set_xlabel("Trial Time (s)")
                        if j == 0:
                            ax.set_ylabel(r"$\beta$")
                    i += 1
    return fig, axes

In [ ]:
encoders = {"full": encoder, "mb": encoder_mb, "mf": encoder_mf}
bw_stats = {k: get_bw_stats(e) for k, e in encoders.items()}
plot_bw_traces(encoders, bw_stats)

In [ ]:
for regr in encoder.tv_keys:
    if regr != "response_prev":
        regr_a = f"{regr}_{tv_vals[regr][0]}"
        regr_b = f"{regr}_{tv_vals[regr][1]}"
        assert all(
            np.isclose(
                bw_stats["full"][0]["DLS"][:, encoder.dm_idxs[regr_a]],
                bw_stats["full"][0]["DLS"][:, encoder.dm_idxs[regr_b]],
            )
        )

## aggregate across sessions

In [ ]:
from sg.models import make_tre, Encoder, StrategyEncoder
from core.data import subject_ids, session_ids
from utils.viz_utils import save_fig
from utils.paths import FIGURES_DIR

for subj_id in ["MR82", "MR83"]:
    print(subj_id)
    fpath = FIGURES_DIR / "time_resolved" / "bweight" / "averaged" / subj_id
    for sess_id in session_ids[np.where(subject_ids == subj_id)[0][0]]:
        print(f">{sess_id}")
        encoder = make_tre(Encoder)(
            subj_id,
            sess_id,
            norm=True,
            stepsize_s=0.1,
        )

        encoder_mb = make_tre(StrategyEncoder)(
            subj_id,
            sess_id,
            norm=True,
            stepsize_s=0.1,
            strategy_filter="mb",
        )

        encoder_mf = make_tre(StrategyEncoder)(
            subj_id,
            sess_id,
            norm=True,
            stepsize_s=0.1,
            strategy_filter="mf",
        )

        encoder.fit_encoder()

        try:
            encoder_mb.fit_encoder()
            encoder_mf.fit_encoder()
        except ValueError:
            continue

        encoders = {"full": encoder, "mb": encoder_mb, "mf": encoder_mf}
        bw_stats = {k: get_bw_stats(e) for k, e in encoders.items()}
        fig, _ = plot_bw_traces(encoders, bw_stats)

        save_fig(fig, fpath, fname=f"{sess_id}.png")